# Chapter 6: Equations in Curvilinear Coordinates

Companion notebook to Section 6 of the study guide (Kundu, Cohen & Dowling 2016, Appendix B.6).
Write your own notes under each heading as you study.

Before committing: **Restart** the kernel, then **Run All**, then **save**, so the notebook runs from top to bottom.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

from ch06_tools import (advect_scalar, convective_acceleration, curl, div,
                        ferguson_church_settling_velocity, grad, laplacian, poiseuille_discharge,
                        poiseuille_velocity, rankine_pressure, rankine_vortex, rouse_number,
                        rouse_profile, stokes_settling_velocity, stokes_stream_function,
                        superelevation, theis_drawdown, vector_laplacian)

FIG_DIR = Path("figures")   # figures worth showing on GitHub
FIG_DIR.mkdir(exist_ok=True)

BLUE, ORANGE = "#2a78d6", "#eb6834"                   # categorical slots 1 and 2
RAMP = ["#86b6ef", "#2a78d6", "#104281"]              # three ordered series
RAMP4 = ["#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]  # four ordered series
GRAY = "#52514e"                                      # reference curves

# Coordinates used throughout: cylindrical (R, phi, z) and spherical (r, theta, phi)
R = sp.Symbol("R", positive=True)
phi, z = sp.symbols("phi z", real=True)
r, theta = sp.symbols("r theta", positive=True)
cyl, h_cyl = (R, phi, z), (1, R, 1)
sph, h_sph = (r, theta, phi), (1, r, r * sp.sin(theta))

## 6.1 Master formulas: everything from the scale factors

With $H = h_1h_2h_3$: $\nabla f = \sum_i \dfrac{\mathbf e_i}{h_i}\dfrac{\partial f}{\partial q_i}$, $\nabla\cdot\mathbf u = \dfrac1H\sum_i\dfrac{\partial}{\partial q_i}\left(\dfrac{H}{h_i}u_i\right)$, $\nabla^2 f = \dfrac1H\sum_i\dfrac{\partial}{\partial q_i}\left(\dfrac{H}{h_i^2}\dfrac{\partial f}{\partial q_i}\right)$,
the curl from a determinant, and $\nabla^2\mathbf u = \nabla(\nabla\cdot\mathbf u) - \nabla\times(\nabla\times\mathbf u)$, the only safe definition of the vector Laplacian.

*My notes:*

In [ ]:
uR, uphi, uz = u = [sp.Function(n)(R, phi, z) for n in ("u_R", "u_phi", "u_z")]
print("cylindrical div u    =", sp.expand(div(u, cyl, h_cyl)))
print("cylindrical omega_z  =", sp.expand(curl(u, cyl, h_cyl)[2]))

VL = vector_laplacian(u, cyl, h_cyl)
print("(lap u)_R   - lap(u_R)   =", sp.simplify(VL[0] - laplacian(uR, cyl, h_cyl)))
print("(lap u)_phi - lap(u_phi) =", sp.simplify(VL[1] - laplacian(uphi, cyl, h_cyl)))

f = sp.Function("f")(r, theta, phi)
print("spherical Laplacian  =", sp.expand(laplacian(f, sph, h_sph)))

## 6.2 Reference formulas

The full cylindrical and spherical lists (gradient, divergence, curl, Laplacians, strain rates, continuity and Navier-Stokes) are in the boxes of the study guide and on Kundu pp. 895-898. All of them follow from the master formulas above.

*My notes:*

## 6.3 Where do the extra terms come from?

They come from differentiating unit vectors that turn: centripetal terms, angular-momentum terms and $\cot\theta$ terms.
Careful: $[(\mathbf u\cdot\nabla)\mathbf u]_R = (\mathbf u\cdot\nabla)u_R - u_\varphi^2/R$, and $(\nabla^2\mathbf u)_R \neq \nabla^2u_R$.

*My notes:*

In [ ]:
print("cylindrical: [(u.grad)u]_i - (u.grad) u_i")
for name, comp, ui in zip(("R", "phi", "z"), convective_acceleration(u, cyl, h_cyl), u):
    print(f"  {name:5s}:", sp.simplify(comp - advect_scalar(ui, u, cyl, h_cyl)))

us = [sp.Function(n)(r, theta, phi) for n in ("u_r", "u_theta", "u_phi")]
print("spherical:   [(u.grad)u]_i - (u.grad) u_i")
for name, comp, ui in zip(("r", "theta", "phi"), convective_acceleration(us, sph, h_sph), us):
    print(f"  {name:5s}:", sp.simplify(comp - advect_scalar(ui, us, sph, h_sph)))

## 6.4 Exact solutions as worked examples

- **Rigid rotation** $u_\varphi = \Omega R$: $\omega_z = 2\Omega$, no strain. **Potential vortex** $u_\varphi = \Gamma/(2\pi R)$: $\omega_z = 0$ for $R\neq0$ (a delta function at the axis). **Rankine vortex:** rigid core, potential outside.
- **Hagen-Poiseuille:** $u_z = \dfrac{G}{4\mu}(a^2 - R^2)$, $Q = \dfrac{\pi Ga^4}{8\mu}$, $\bar u = u_{\max}/2$.
- **Stokes flow past a sphere:** drag $6\pi\mu aU$; settling velocity $w_s = R_sgd^2/(18\nu)$.

*My notes:*

In [ ]:
a, Um = 10.0, 10.0                           # a dust devil: core radius 10 m, peak wind 10 m/s
Rg = np.linspace(0.01 * a, 5 * a, 2000)
rigid, potential = Um * Rg / a, Um * a / Rg
rank = rankine_vortex(Rg, a, Um)
omega = np.gradient(Rg * rank, Rg, edge_order=2) / Rg   # omega_z = (1/R) d(R u_phi)/dR

fig, axes = plt.subplots(1, 3, figsize=(15, 3.9))
axes[0].plot(Rg / a, rank / Um, color=GRAY, lw=3, alpha=0.5, label="Rankine")
axes[0].plot(Rg / a, rigid / Um, "--", color=BLUE, label="rigid rotation")
axes[0].plot(Rg / a, potential / Um, "--", color=ORANGE, label="potential vortex")
axes[0].set(ylim=(0, 2), xlabel="R / a", ylabel="u_phi / U_m", title="(a) azimuthal velocity")
axes[0].legend(fontsize=8)

axes[1].plot(Rg / a, omega * a / Um, color=GRAY, lw=3, alpha=0.5, label="Rankine")
axes[1].axhline(2, color=BLUE, ls="--", label="rigid rotation: 2 Omega")
axes[1].axhline(0, color=ORANGE, ls="--", label="potential vortex: 0 (spike at R = 0)")
axes[1].set(ylim=(-0.2, 2.4), xlabel="R / a", ylabel="omega_z a / U_m", title="(b) vorticity")
axes[1].legend(fontsize=8, loc="center right")

axes[2].plot(Rg / a, rankine_pressure(Rg, a, Um, 1.0) / Um**2, color=GRAY, lw=2)
axes[2].set(xlabel="R / a", ylabel="(p - p_inf) / (rho U_m^2)", title="(c) Rankine vortex: low pressure core")
fig.tight_layout()
fig.savefig(FIG_DIR / "ch06_vortices.png", dpi=200)
plt.show()

print(f"circulation {2 * np.pi * a * Um:.0f} m^2/s, core vorticity {2 * Um / a:.1f} 1/s, "
      f"central pressure deficit {-rankine_pressure(0.0, a, Um, 1.2):.0f} Pa (air)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))

# (a) Poiseuille flow in a soil root channel
a_p, G, mu, nu = 2e-4, 1000 * 9.81, 1.0e-3, 1.0e-6   # radius 0.2 mm, unit hydraulic gradient
Rp = np.linspace(-a_p, a_p, 401)
u_max = poiseuille_velocity(0.0, a_p, G, mu)
Q = poiseuille_discharge(a_p, G, mu)
mean = Q / (np.pi * a_p**2)
axes[0].plot(poiseuille_velocity(Rp, a_p, G, mu) / u_max, Rp / a_p, color=BLUE)
axes[0].axvline(mean / u_max, color=GRAY, ls=":", lw=1.5)
axes[0].text(mean / u_max + 0.02, -0.9, "mean = u_max / 2", fontsize=8, color=GRAY)
axes[0].set(xlim=(0, 1.1), xlabel="u_z / u_max", ylabel="R / a", title="(a) Poiseuille: parabolic profile")
print(f"root channel: Q = {Q:.2e} m^3/s, mean {mean * 1e3:.0f} mm/s, Re = {2 * a_p * mean / nu:.0f}; "
      f"doubling the radius multiplies Q by {poiseuille_discharge(2 * a_p, G, mu) / Q:.0f}")

# (b) Stokes flow past a sphere: streamlines, compared with the undisturbed stream
x = np.linspace(-3, 3, 401)
X, Z = np.meshgrid(x, x)                     # meridional plane, sphere of radius 1
rr = np.hypot(X, Z)
th = np.arccos(np.clip(Z / np.maximum(rr, 1e-12), -1, 1))
psi = np.ma.masked_where(rr < 1, stokes_stream_function(np.maximum(rr, 1.0), th, 1.0, 1.0))
levels = [0.02, 0.1, 0.25, 0.5, 0.8, 1.2, 1.7, 2.3, 3.0, 3.8]
axes[1].contour(X, Z, 0.5 * X**2, levels=levels, colors=GRAY, linewidths=0.6, linestyles="dotted")
axes[1].contour(X, Z, psi, levels=levels, colors=BLUE, linewidths=1.0, linestyles="solid")
axes[1].add_patch(plt.Circle((0, 0), 1, color="#cde2fb"))
axes[1].set(aspect="equal", xlabel="x / a", ylabel="z / a (flow upward)",
            title="(b) Stokes flow (blue) vs. undisturbed stream (dotted)")
fig.tight_layout()
fig.savefig(FIG_DIR / "ch06_pipe_and_stokes.png", dpi=200)
plt.show()

In [ ]:
# Stokes' solution checked with the master formulas (no hand-written spherical operators)
a_s, U, mu_s = sp.symbols("a U mu", positive=True)
s, c = sp.sin(theta), sp.cos(theta)
u_st = [U * c * (1 - 3 * a_s / (2 * r) + a_s**3 / (2 * r**3)),
        -U * s * (1 - 3 * a_s / (4 * r) - a_s**3 / (4 * r**3)), 0]
p_st = -sp.Rational(3, 2) * mu_s * a_s * U * c / r**2
print("div u              =", sp.simplify(div(u_st, sph, h_sph)))                        # 0
print("no slip at r = a   =", [sp.simplify(ui.subs(r, a_s)) for ui in u_st[:2]])          # [0, 0]
VL_st, gp = vector_laplacian(u_st, sph, h_sph), grad(p_st, sph, h_sph)
print("grad p - mu lap u  =", [sp.simplify(gp[i] - mu_s * VL_st[i]) for i in range(3)])  # [0, 0, 0]
print("vorticity (phi)    =", sp.simplify(curl(u_st, sph, h_sph)[2]))

## 6.5 Research connection: curvature, grains and wells

- **Meanders:** $-u_\varphi^2/R \approx -\rho^{-1}\partial p/\partial R$ gives a superelevation $\Delta\zeta \approx U^2B/(gR_c)$ and a helical secondary flow that builds point bars.
- **Settling and suspension:** Stokes' law for fine grains, Ferguson and Church (2004) for natural grains; Rouse number $P = w_s/(\kappa u_*)$ and profile $c/c_a = \{[(h-z)/z]\,[a/(h-a)]\}^P$.
- **Wells:** Thiem (steady, logarithmic) and Theis (transient, similarity variable $r^2S/4Tt$).
- **Domes:** the $1/R$ geometry makes steady domes half as high as ridges.

*My notes:*

In [ ]:
U_bend, B = 1.2, 50.0                         # mean velocity [m/s], channel width [m]
for ratio in (2, 3, 5, 10):
    print(f"Rc / B = {ratio:2d}: superelevation {100 * superelevation(U_bend, B, ratio * B):.1f} cm")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

# (a) settling velocity of quartz grains in water
d = np.logspace(-6, -2.5, 200)                # 1 um to 3 mm
ws_st, ws_fc = stokes_settling_velocity(d), ferguson_church_settling_velocity(d)
axes[0].loglog(d * 1e6, ws_st, "--", color=GRAY, label="Stokes' law")
axes[0].loglog(d * 1e6, ws_fc, color=BLUE, label="Ferguson & Church (2004)")
d1 = d[np.argmin(np.abs(ws_st * d / 1e-6 - 1))]
axes[0].axvline(d1 * 1e6, color=GRAY, lw=0.8, ls=":")
axes[0].text(d1 * 1e6 * 1.15, 1e-6, "Re_p = 1", fontsize=8, color=GRAY)
axes[0].set(xlabel="grain diameter (um)", ylabel="settling velocity (m/s)",
            title="(a) Stokes' law fails for sand")
axes[0].legend(fontsize=8)

# (b) Rouse profiles
h, a_ref = 1.0, 0.05
zz = np.linspace(a_ref, 0.995 * h, 300)
for P, color in zip((0.05, 0.3, 1.0, 2.5), RAMP4):
    axes[1].plot(rouse_profile(zz, h, a_ref, P), zz / h, color=color, label=f"P = {P}")
axes[1].set(xlim=(0, 1.02), xlabel="c / c_a", ylabel="z / h", title="(b) Rouse profiles: P sets the shape")
axes[1].legend(fontsize=8)

# (c) Theis drawdown cone around a pumping well
Q_w, T, S = 0.005, 2e-3, 2e-4                 # 5 L/s; transmissivity m^2/s; storativity
rw = np.logspace(-1, 4, 300)
for t_s, label, color in zip((3600, 86400, 30 * 86400), ("1 hour", "1 day", "30 days"), RAMP):
    axes[2].semilogx(rw, theis_drawdown(rw, t_s, Q_w, T, S), color=color, label=label)
axes[2].invert_yaxis()
axes[2].set(xlabel="distance from well (m)", ylabel="drawdown (m)",
            title="(c) Theis: the cone of depression spreads")
axes[2].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "ch06_grains_and_wells.png", dpi=200)
plt.show()

u_star = 0.04                                 # shear velocity [m/s]
for d_um in (20, 125, 500):
    ws = ferguson_church_settling_velocity(d_um * 1e-6)
    print(f"d = {d_um:3d} um: w_s = {ws * 1e3:6.2f} mm/s, Rouse number {rouse_number(ws, u_star):.2f}")

## Exercises 6.1-6.9

*Work space: solve each exercise here, then check against Appendix C of the study guide.*